In [ ]:
import sys
from pathlib import Path

current_dir = Path.cwd().resolve()

# Find 'src' by walking up the directory tree
src_path = None
for parent in [current_dir] + list(current_dir.parents):
    candidate = parent / 'src'
    if candidate.exists() and candidate.is_dir():
        src_path = candidate
        break

if src_path:
    if str(src_path) not in sys.path:
        sys.path.append(str(src_path))
    print(f"Success: Linked to src at {src_path}")
    
    # Import
    try:
        from config import *
        print(f"Environment linked. Data dir: {DATA_DIR}")    
    except ImportError as e:
        print(f"CRITICAL: Found src but could not import config. {e}")
else:
    print("CRITICAL ERROR: Could not find 'src' directory in any parent folder.")
    print(f"Current search path: {current_dir}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from plotting import *
setup_plotting_style()
print(f"saving figures:{SAVE_FIGURES}")

In [ ]:
import numpy as np
import pandas as pd

NB_ID = "22"

In [ ]:
def load_spot_data():
    print(f"Loading {MIT_SPOT_X.name}...")
    X = np.load(MIT_SPOT_X)
    Y = np.load(MIT_SPOT_Y)
    df = pd.read_csv(MIT_SPOT_DATASET_METADATA_FILE)
    
    return X, Y, df

def load_barrage_data():    
    print(f"Loading {MIT_BARRAGE_X.name}...")
    X = np.load(MIT_BARRAGE_X)
    Y = np.load(MIT_BARRAGE_Y)
    df = pd.read_csv(MIT_BARRAGE_DATASET_METADATA_FILE)
    
    return X, Y, df

In [ ]:
X_barr, Y_barr, df_barr = load_barrage_data()
X_spot, Y_spot, df_spot = load_spot_data()

max_barrage = np.max(np.abs(X_barr))
max_spot = np.max(np.abs(X_spot))

GLOBAL_MAX = max(max_barrage, max_spot)

print(f"\n--- Calibration ---")
print(f"Barrage Raw Max: {max_barrage:.4f}")
print(f"Spot Raw Max:    {max_spot:.4f}")
print(f"----------------------------")
print(f"MASTER SCALING FACTOR (M): {GLOBAL_MAX:.4f}")
print(f"----------------------------")

In [ ]:
def save_standardized(X, Y, X_path, Y_path, global_max):
    # Scale to [0, 1] range
    X_norm = (X / global_max).astype(np.complex64)
    Y_norm = (Y / global_max).astype(np.complex64)
    
    
    np.save(X_path, X_norm)
    np.save(Y_path, Y_norm)
    
    print(f"saved (Max: {np.max(np.abs(X_norm)):.4f}) to {PROCESSED_DIR}")


save_standardized(X_barr, Y_barr, MIT_BARRAGE_X_NORMALIZED, MIT_BARRAGE_Y_NORMALIZED, GLOBAL_MAX)
save_standardized(X_spot, Y_spot, MIT_SPOT_X_NORMALIZED, MIT_SPOT_Y_NORMALIZED, GLOBAL_MAX)


with open( MASTER_SCALING_FACTORS_FILE, "w") as f:
    f.write(str(GLOBAL_MAX))

In [ ]:
def plot_distribution(X_data, label, color, limit=100000):
    # Flatten absolute magnitudes
    mags = np.abs(X_data).ravel()
    
    # Randomly sample if data is too big
    if len(mags) > limit:
        indices = np.random.choice(len(mags), size=limit, replace=False)
        mags_sampled = mags[indices]
    else:
        mags_sampled = mags
        
    # Plot using normalized data (divided by Global Max)
    sns.kdeplot(mags_sampled / GLOBAL_MAX, label=f"{label} (Sampled)", fill=True, color=color, alpha=0.3)

plt.figure(figsize=(10, 6))

# Plot subsets safely
plot_distribution(X_barr, "Barrage", "C0")
plot_distribution(X_spot, "Spot", "C1")

plt.axvline(1.0, color='C3', linestyle='--', label="NN Limit (1.0)")
plt.title(f"Final Standardized Distribution (M={GLOBAL_MAX:.4f})")
plt.xlabel("Normalized Magnitude")
plt.legend()
plt.grid(True, alpha=0.3)

save_plot("Standardized Distribution", machine_id="B",nb_id=NB_ID,fig_id="01")
plt.show()